# Experiment 2: Context Size (Chunk Size)

## Research Question
How much contextual information does the LLM require to detect intertextuality effectively?

## Experimental Design

This experiment investigates the effect of the sliding window context size.

### Fixed Parameters
- Model: gpt-5.4-mini
- Prompt: best prompt from Experiment 1 -> for gpt-5.4-mini it is maximum prompt  (minimal prompt for gpt-4o-mini)
- Overlap: 20
- Few-shot examples: 0

### Variable Parameter
- chunk_size ∈ {120,360,680,720,900}

### Evaluation

The experiment is executed across five folds.  
Performance is measured using precision, recall, and F1-score.

In [1]:
import sys
project_src = r"C:\Users\sophi\Desktop\Master\CIS_5.Semester\Thesis\locisimiles\src"

# lokalen Pfad *vor* site-packages setzen
sys.path.insert(0, project_src)

import locisimiles
print("Jetzt geladen aus:", locisimiles.__file__)

#imports
import openai
from unittest.mock import patch
import locisimiles as ls
from locisimiles.evaluator import IntertextEvaluator
#from locisimiles.llm_pipeline_new import LLMOnlyClassificationPipeline
from locisimiles.llm_pipeline_sliding_windows import LLMOnlySlidingWindowPipeline, pretty_print, save_final_results, load_final_results
from locisimiles.document import Document, TextSegment
#from locisimiles.llm_pipeline_new import pretty_print
from locisimiles.llm_pipeline_sliding_windows import convert_llm_results, fixed_sliding_windows, convert_llm_results_coverage_fair, pad_segments
import pandas as pd
import csv
from pathlib import Path
from locisimiles.evaluator import IntertextEvaluator
import random
import tempfile
import time

# load key
from dotenv import load_dotenv
load_dotenv()

Jetzt geladen aus: C:\Users\sophi\Desktop\Master\CIS_5.Semester\Thesis\locisimiles\src\locisimiles\__init__.py


c:\Users\sophi\anaconda3\envs\py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# =========================
# DATA PATH
# =========================
DATA_ROOT = Path("../Sampled_Documents/folds")

N_FOLDS = 5

In [3]:
def load_fold(
    fold_id: int,
    *,
    root: Path = DATA_ROOT,
):
    """
    Load query document, source document and ground truth for one fold.
    """

    fold_dir = root / f"fold_{fold_id}" / "evaluation"

    if not fold_dir.exists():
        raise FileNotFoundError(f"Fold {fold_id} not found at {fold_dir}")

    query_doc = Document(fold_dir / "query_document.csv")
    source_doc = Document(fold_dir / "source_document.csv")
    ground_truth_path = fold_dir / "ground_truth.csv"

    return {
        "query": query_doc,
        "source": source_doc,
        "ground_truth": ground_truth_path,
    }

In [ ]:
# =========================
# FIXED PARAMETERS
# =========================

MODEL = "gpt-5.4-nano"
BEST_PROMPT = "zero_shot_maximum" #replace with result from Experiment 1-> maximum for gpt-5.4-mini

OVERLAP = 20
NUM_FEW_SHOT = 0

# =========================
# VARIABLE/EXPERIMENT PARAMETER
# =========================
#run not on 120 because have that from experiment 1
#CHUNK_SIZES = [360,680,720,900]
#CHUNK_SIZES = [120,360,680,720,900]
CHUNK_SIZES = [120]

# ==============================
# OUTPUT
# ==============================

RESULT_DIR = Path("final_results/experiment_2_chunk_size")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
# =========================
# RUN EXPERIMENT
# =========================
for chunk_size in CHUNK_SIZES:

    print("\n====================================")
    print(f"Running experiment for chunk_size: {chunk_size}")
    print("====================================")

    pipeline = LLMOnlySlidingWindowPipeline(
        openai_model=MODEL,
        prompt_type=BEST_PROMPT,
        chunk_size=chunk_size,
        overlap=OVERLAP,
        num_few_shot_examples=NUM_FEW_SHOT,
    )

    for fold_id in range(N_FOLDS):

        print(f"\nRunning fold {fold_id}")

        fold = load_fold(fold_id)

        checkpoint_path = RESULT_DIR / f"checkpoint_fold_{fold_id}_chunkSize{chunk_size}.json"
        
        start_time = time.perf_counter()

        results = pipeline.run(
            query=fold["query"],
            source=fold["source"],
            checkpoint_path=checkpoint_path,
            resume=True,
        )
        
        end_time = time.perf_counter()
        runtime = end_time - start_time
        print(f"Fold {fold_id} runtime: {runtime:.2f} seconds")

        out_path = RESULT_DIR / f"fold_{fold_id}_chunk_{chunk_size}.json"

        save_final_results(
            results,
            path=out_path,
            meta={
                "model": pipeline.openai_model,
                "prompt_type": BEST_PROMPT,
                "chunk_size": pipeline.chunk_size,
                "num_few_shot_examples": pipeline.num_few_shot_examples,
                "overlap": pipeline.overlap,
                "runtime_seconds": runtime,
            },
        )

        print(f"Saved results to {out_path}")


Running experiment for chunk_size: 120

Running fold 0
Total source windows: 10


Processing source windows:   0%|          | 0/10 [00:00<?, ?it/s]

Processing source windows: 100%|██████████| 10/10 [13:56<00:00, 83.70s/it]


Fold 0 runtime: 837.07 seconds
Saved results to final_results\experiment_2_chunk_size\fold_0_chunk_120.json

Running fold 1
Total source windows: 10


Processing source windows: 100%|██████████| 10/10 [13:44<00:00, 82.42s/it]


Fold 1 runtime: 824.22 seconds
Saved results to final_results\experiment_2_chunk_size\fold_1_chunk_120.json

Running fold 2
Total source windows: 10


... (2 outputs left out)

Processing source windows: 100%|██████████| 10/10 [16:47<00:00, 100.71s/it]


Fold 3 runtime: 1007.13 seconds
Saved results to final_results\experiment_2_chunk_size\fold_3_chunk_120.json

Running fold 4
Total source windows: 10


Processing source windows: 100%|██████████| 10/10 [13:50<00:00, 83.05s/it]


Fold 4 runtime: 830.53 seconds
Saved results to final_results\experiment_2_chunk_size\fold_4_chunk_120.json


In [11]:
# =========================
# EVALUATION for size 120
# =========================
all_scores_120 = []

for chunk_size in CHUNK_SIZES:

    print("\n===============================")
    print(f"Evaluating chunk_size: {chunk_size}")
    print("===============================")

    for fold_id in range(N_FOLDS):

        print(f"\nEvaluating fold {fold_id}")

        fold = load_fold(fold_id)

        result_file = RESULT_DIR / f"fold_{fold_id}_chunk_{chunk_size}.json"

        raw_results = load_final_results(result_file)
        
        pipeline = LLMOnlySlidingWindowPipeline(
            openai_model=MODEL,
            prompt_type=BEST_PROMPT,
            chunk_size=chunk_size,
            overlap=OVERLAP,
            num_few_shot_examples=NUM_FEW_SHOT,
        )


        source_segments = pad_segments(
            list(fold["source"].segments.values()),
            chunk_size,
            prefix="S"
        )

        query_segments = pad_segments(
            list(fold["query"].segments.values()),
            chunk_size,
            prefix="Q"
        )

        source_windows = fixed_sliding_windows(
            source_segments,
            chunk_size,
            pipeline.stride
        )

        query_windows = fixed_sliding_windows(
            query_segments,
            chunk_size,
            pipeline.stride
        )

        #print("DEBUG:", source_windows, query_windows)
        full_results = convert_llm_results_coverage_fair(
            raw_results,
            fold["source"],
            fold["query"],
            source_windows,
            query_windows,
            threshold=0.0
        )

        wrapped_pipeline = type(
            "DummyPipeline",
            (),
            {"run": lambda self, **kwargs: full_results}
        )()

        evaluator = IntertextEvaluator(
            query_doc=fold["query"],
            source_doc=fold["source"],
            ground_truth_csv=fold["ground_truth"],
            pipeline=wrapped_pipeline,
            top_k=1000,
            threshold=0.5,
        )

        macro_scores = evaluator.evaluate(average="macro")
        micro_scores = evaluator.evaluate(average="micro")

        print("Macro:", macro_scores)
        print("Micro:", micro_scores)
        
        macro_row = macro_scores.iloc[0]
        micro_row = micro_scores.iloc[0]

        all_scores_120.append({
            "chunk_size": chunk_size,
            "fold": fold_id,

            "macro_precision": macro_row["precision"],
            "macro_recall": macro_row["recall"],
            "macro_f1": macro_row["f1"],
            "macro_accuracy": macro_row["accuracy"],
            "macro_fpr": macro_row["fpr"],
            "macro_fnr": macro_row["fnr"],
            "macro_smr": macro_row["smr"],
            "macro_tp": macro_row["tp"],
            "macro_fp": macro_row["fp"],
            "macro_fn": macro_row["fn"],
            "macro_tn": macro_row["tn"],

            "micro_precision": micro_row["precision"],
            "micro_recall": micro_row["recall"],
            "micro_f1": micro_row["f1"],
            "micro_accuracy": micro_row["accuracy"],
            "micro_fpr": micro_row["fpr"],
            "micro_fnr": micro_row["fnr"],
            "micro_smr": micro_row["smr"],
            "micro_tp": micro_row["tp"],
            "micro_fp": micro_row["fp"],
            "micro_fn": micro_row["fn"],
            "micro_tn": micro_row["tn"],
        })


Evaluating chunk_size: 120

Evaluating fold 0

PAIR: <hier. epist. 123.3.1.1> || <cic. ac. 2.58.2>
  labels: [0, 1]
  sum(labels): 1
  len(labels): 2
  coverage_source: 1
  coverage_query: 2
  max_labels: 2
  prob: 0.5000
  observed/max: 1.0000
--------------------------------------------------

PAIR: <hier. epist. 117.12.1.1> || <cic. att. 12.1.2.5>
  labels: [0, 1]
  sum(labels): 1
  len(labels): 2
  coverage_source: 1
  coverage_query: 2
  max_labels: 2
  prob: 0.5000
  observed/max: 1.0000
--------------------------------------------------

PAIR: <hier. epist. 107.9.2.3> || <cic. att. 13.20.4.2>
  labels: [0, 1]
  sum(labels): 1
  len(labels): 2
  coverage_source: 1
  coverage_query: 2
  max_labels: 2
  prob: 0.5000
  observed/max: 1.0000
--------------------------------------------------

PAIR: <hier. epist. 6.2.1.3> || <hor. sat. 1.3.1.1>
  labels: [1, 0]
  sum(labels): 1
  len(labels): 2
  coverage_source: 2
  coverage_query: 1
  max_labels: 2
  prob: 0.5000
  observed/max: 1.0

In [ ]:
# =========================
# EVALUATION
# =========================
all_scores = []

for chunk_size in CHUNK_SIZES:

    print("\n===============================")
    print(f"Evaluating chunk_size: {chunk_size}")
    print("===============================")

    for fold_id in range(N_FOLDS):

        print(f"\nEvaluating fold {fold_id}")

        fold = load_fold(fold_id)

        result_file = RESULT_DIR / f"fold_{fold_id}_chunk_{chunk_size}.json"

        raw_results = load_final_results(result_file)
        
        pipeline = LLMOnlySlidingWindowPipeline(
            openai_model=MODEL,
            prompt_type=BEST_PROMPT,
            chunk_size=chunk_size,
            overlap=OVERLAP,
            num_few_shot_examples=NUM_FEW_SHOT,
        )


        source_segments = pad_segments(
            list(fold["source"].segments.values()),
            chunk_size,
            prefix="S"
        )

        query_segments = pad_segments(
            list(fold["query"].segments.values()),
            chunk_size,
            prefix="Q"
        )

        source_windows = fixed_sliding_windows(
            source_segments,
            chunk_size,
            pipeline.stride
        )

        query_windows = fixed_sliding_windows(
            query_segments,
            chunk_size,
            pipeline.stride
        )

        #print("DEBUG:", source_windows, query_windows)
        full_results = convert_llm_results_coverage_fair(
            raw_results,
            fold["source"],
            fold["query"],
            source_windows,
            query_windows,
            threshold=0.0
        )

        wrapped_pipeline = type(
            "DummyPipeline",
            (),
            {"run": lambda self, **kwargs: full_results}
        )()

        evaluator = IntertextEvaluator(
            query_doc=fold["query"],
            source_doc=fold["source"],
            ground_truth_csv=fold["ground_truth"],
            pipeline=wrapped_pipeline,
            top_k=1000,
            threshold=0.5,
        )

        macro_scores = evaluator.evaluate(average="macro")
        micro_scores = evaluator.evaluate(average="micro")

        print("Macro:", macro_scores)
        print("Micro:", micro_scores)
        
        macro_row = macro_scores.iloc[0]
        micro_row = micro_scores.iloc[0]

        all_scores.append({
            "chunk_size": chunk_size,
            "fold": fold_id,

            "macro_precision": macro_row["precision"],
            "macro_recall": macro_row["recall"],
            "macro_f1": macro_row["f1"],
            "macro_accuracy": macro_row["accuracy"],
            "macro_fpr": macro_row["fpr"],
            "macro_fnr": macro_row["fnr"],
            "macro_smr": macro_row["smr"],
            "macro_tp": macro_row["tp"],
            "macro_fp": macro_row["fp"],
            "macro_fn": macro_row["fn"],
            "macro_tn": macro_row["tn"],

            "micro_precision": micro_row["precision"],
            "micro_recall": micro_row["recall"],
            "micro_f1": micro_row["f1"],
            "micro_accuracy": micro_row["accuracy"],
            "micro_fpr": micro_row["fpr"],
            "micro_fnr": micro_row["fnr"],
            "micro_smr": micro_row["smr"],
            "micro_tp": micro_row["tp"],
            "micro_fp": micro_row["fp"],
            "micro_fn": micro_row["fn"],
            "micro_tn": micro_row["tn"],
        })


Evaluating chunk_size: 360

Evaluating fold 0

PAIR: <lact. ira 13.5.1> || <verg. georg. 1.289.1>
  labels: [0, 1]
  sum(labels): 1
  len(labels): 2
  coverage_source: 1
  coverage_query: 2
  max_labels: 2
  prob: 0.5000
  observed/max: 1.0000
--------------------------------------------------

PAIR: <hier. in zach. 1454d.1> || <hor. epist. 1.2.39.1>
  labels: [1]
  sum(labels): 1
  len(labels): 1
  coverage_source: 1
  coverage_query: 1
  max_labels: 1
  prob: 1.0000
  observed/max: 1.0000
--------------------------------------------------

PAIR: <hier. epist. 27.3.1.1> || <hor. ars. 21.1>
  labels: [1]
  sum(labels): 1
  len(labels): 1
  coverage_source: 1
  coverage_query: 1
  max_labels: 1
  prob: 1.0000
  observed/max: 1.0000
--------------------------------------------------

PAIR: <lact. inst. 3.17.10.1> || <lucr. 2.1096.1>
  labels: [1]
  sum(labels): 1
  len(labels): 1
  coverage_source: 1
  coverage_query: 1
  max_labels: 1
  prob: 1.0000
  observed/max: 1.0000
-------------

In [12]:
all_scores_120

[{'chunk_size': 120,
  'fold': 0,
  'macro_precision': np.float64(0.03930985414443259),
  'macro_recall': np.float64(0.044823906083244394),
  'macro_f1': np.float64(0.04108858057630736),
  'macro_accuracy': np.float64(0.9999017657902396),
  'macro_fpr': np.float64(1.8191520325992043e-05),
  'macro_fnr': np.float64(8.0042689434365e-05),
  'macro_smr': np.float64(9.823420976035703e-05),
  'macro_tp': np.float64(43.0),
  'macro_fp': np.float64(15.0),
  'macro_fn': np.float64(66.0),
  'macro_tn': np.float64(824436.0),
  'micro_precision': np.float64(0.7413793103448276),
  'micro_recall': np.float64(0.3944954128440367),
  'micro_f1': np.float64(0.5149700598802396),
  'micro_accuracy': np.float64(0.9999017657902396),
  'micro_fpr': np.float64(1.8191520325992043e-05),
  'micro_fnr': np.float64(8.004268943436499e-05),
  'micro_smr': np.float64(9.823420976035704e-05),
  'micro_tp': np.float64(43.0),
  'micro_fp': np.float64(15.0),
  'micro_fn': np.float64(66.0),
  'micro_tn': np.float64(824436.

In [13]:
# =========================
# RESULTS AGGREGATION
# =========================
df = pd.DataFrame(all_scores+all_scores_120)

summary = (
    df.groupby("chunk_size")
    .mean(numeric_only=True)
    .sort_values("macro_f1", ascending=False)
)

summary

,fold,macro_precision,macro_recall,macro_f1,macro_accuracy,macro_fpr,macro_fnr,macro_smr,macro_tp,macro_fp,...,micro_recall,micro_f1,micro_accuracy,micro_fpr,micro_fnr,micro_smr,micro_tp,micro_fp,micro_fn,micro_tn
chunk_size,,,,,,,,,,,,,,,,,,,,,
120,2.0,0.040259,0.043693,0.041137,0.999902,0.000017,0.000081,0.000098,41.8,13.6,...,0.384166,0.509672,0.999902,0.000017,0.000081,0.000098,41.8,13.6,67.0,823029.6
360,2.0,0.027213,0.027569,0.027249,0.999895,0.000005,0.000100,0.000105,26.2,4.0,...,0.240877,0.373546,0.999895,0.000005,0.000100,0.000105,26.2,4.0,82.6,823039.2
720,2.0,0.021605,0.022458,0.021818,0.999889,0.000004,0.000106,0.000111,21.2,3.6,...,0.194869,0.316615,0.999889,0.000004,0.000106,0.000111,21.2,3.6,87.6,823039.6
680,2.0,0.018392,0.018819,0.018463,0.999886,0.000003,0.000111,0.000114,17.8,2.8,...,0.163660,0.273451,0.999886,0.000003,0.000111,0.000114,17.8,2.8,91.0,823040.4
900,2.0,0.017696,0.018052,0.017696,0.999884,0.000005,0.000111,0.000116,17.2,4.2,...,0.158223,0.261260,0.999884,0.000005,0.000111,0.000116,17.2,4.2,91.6,823039.0


In [15]:
df

,chunk_size,fold,macro_precision,macro_recall,macro_f1,macro_accuracy,macro_fpr,macro_fnr,macro_smr,macro_tp,...,micro_recall,micro_f1,micro_accuracy,micro_fpr,micro_fnr,micro_smr,micro_tp,micro_fp,micro_fn,micro_tn
0,360,0,0.022946,0.023479,0.023123,0.999892,0.000004,0.000104,0.000108,23.0,...,0.211009,0.340741,0.999892,0.000004,0.000104,0.000108,23.0,3.0,86.0,824448.0
1,360,1,0.024064,0.024064,0.023886,0.999891,0.000005,0.000105,0.000109,23.0,...,0.211009,0.338235,0.999891,0.000005,0.000105,0.000109,23.0,4.0,86.0,822687.0
2,360,2,0.037709,0.037887,0.037531,0.999907,0.000005,0.000089,0.000093,36.0,...,0.330275,0.483221,0.999907,0.000005,0.000089,0.000093,36.0,4.0,73.0,824447.0
3,360,3,0.020430,0.020430,0.020430,0.999889,0.000001,0.000110,0.000111,19.0,...,0.174312,0.294574,0.999889,0.000001,0.000110,0.000111,19.0,1.0,90.0,818290.0
4,360,4,0.030917,0.031983,0.031272,0.999896,0.000010,0.000094,0.000104,30.0,...,0.277778,0.410959,0.999896,0.000010,0.000094,0.000104,30.0,8.0,78.0,825324.0
5,680,0,0.011206,0.012807,0.011740,0.999875,0.000007,0.000118,0.000125,12.0,...,0.110092,0.188976,0.999875,0.000007,0.000118,0.000125,12.0,6.0,97.0,824445.0
6,680,1,0.011765,0.012834,0.012121,0.999878,0.000004,0.000118,0.000122,12.0,...,0.110092,0.193548,0.999878,0.000004,0.000118,0.000122,12.0,3.0,97.0,822688.0
7,680,2,0.023479,0.022946,0.023123,0.999892,0.000002,0.000106,0.000108,22.0,...,0.201835,0.330827,0.999892,0.000002,0.000106,0.000108,22.0,2.0,87.0,824449.0
8,680,3,0.023656,0.023118,0.023297,0.999891,0.000002,0.000106,0.000109,22.0,...,0.201835,0.330827,0.999891,0.000002,0.000106,0.000109,22.0,2.0,87.0,818289.0
9,680,4,0.021855,0.022388,0.022033,0.999893,0.000001,0.000105,0.000107,21.0,...,0.194444,0.323077,0.999893,0.000001,0.000105,0.000107,21.0,1.0,87.0,825331.0


In [14]:
# =========================
# SAVE RESULT EVALUATION
# =========================
summary_path = RESULT_DIR / "chunk_size_experiment_summary.csv"

summary.to_csv(summary_path)
df.to_csv(RESULT_DIR / "chunk_size_experiment_results.csv", index=False)

print("Saved summary to", summary_path)

Saved summary to final_results\experiment_2_chunk_size\chunk_size_experiment_summary.csv
